# Chess-Coach Agent — **Gemma 4 E4B** QLoRA on Kaggle **2×T4**

LoRA adapter on the v1.2 agentic-harness corpus (skills + tools + the `python` verify
tool; fast/think/auto). Download the adapter → serve as GGUF on the RTX 4060.

## Why 2×T4
E4B 4-bit @ seq 1664 does **not** fit one 16 GB T4 (forward alone OOMs). With two T4s the
trainer auto-sets `device_map="balanced"` — **shards** the model across both (~28 GB,
model-parallel). NOT DDP (DDP replicates per GPU and OOMs). Single T4 → use the E2B notebook.

## Kaggle setup
1. **Settings → Accelerator → GPU T4 ×2** (required). 2. **Internet → On.**
3. **Add-ons → Secrets → `HF_TOKEN`** (HF read). Accept the license once: https://huggingface.co/unsloth/gemma-4-E4B-it
4. Run top→bottom. After Cell 4 (deps), restart if prompted, then continue from Cell 5.
   **Cell 6.5 (fit test) is the go/no-go.**

## This run = Stage 0 DE-RISK (~250 steps)
Goal: confirm loss falls (~0.1) AND that E4B does **verification-as-tool-use** (calls the
`python` tool, reads stdout, narrates that value). 250 steps fits ONE 12h session. The FULL
run (≈1 epoch) comes after, via resume.

## Multi-session resume (12h ceiling / multiple accounts)
- Trainer checkpoints every `SAVE_EVERY` steps to `runs/OUTPUT/checkpoint`, and saves once
  more on a timeout/crash. `/kaggle/working` persists to Output even if the kernel is killed.
- **Continue on any account:** download the Cell-8 zip → add it as a Kaggle Dataset →
  next session set `RESUME=True` + `RESUME_DATASET`, run Cells 1–6 → **6.6** → 7. Repeat
  until `MAX_STEPS` (raise it to ~4700 for ≈1 epoch).

## OOM ladder (never cut seq — it's the data floor)
RANK 16→8 → keep TARGETS attn-only → (last resort) the E2B notebook.


In [ ]:
# Cell 1 — config (E4B QLoRA on Kaggle 2×T4, model-parallel sharded)
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"   # Gemma-4-native; on 2 GPUs the trainer auto-sets device_map="balanced".

HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False  # E4B MUST be 4-bit

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"
DATA_STEM = "v1_2_lean"   # M1 lean subset (better balance, ~26%% smaller). Full = "v1_2".

# SEQ = data floor (max row 1655, median 1291). Below 1664 truncates >50% of finals. Fixed.
MAX_SEQ = 1472   # DE-RISK floor: V1_R python-verify rows max out at 1469, so 1472 keeps
                 # them 100%% intact and fits ONE T4. FULL run needs 1664 -> forced 2-GPU
                 # split (set CHESS_GPU_CAP_GIB) or an L4. Single T4 can't do E4B@1664.
TARGETS = "attn-only"    # OOM ladder: attn-only + rank 16 -> rank 8 -> never cut seq.
RANK = 16
GRAD_ACCUM = 16

# --- Stage 0 DE-RISK now: ~250 steps shows loss ~0.1 + whether E4B does python-verify. ---
# For the FULL run later, raise MAX_STEPS to ~4700 (≈1 epoch) and resume across sessions.
MAX_STEPS = 250
EVAL_EVERY = 100
MAX_VAL = 128
SAVE_EVERY = 50          # checkpoint adapter+optimizer every 50 steps (crash/timeout-safe + resume)

# --- Multi-session resume (M2): set True on the 2nd+ session (any Kaggle account). ---
# Flow: session ends -> Cell 8 zips runs/OUTPUT -> download. Next session: add that zip as a
# Kaggle Dataset, set RESUME=True + RESUME_DATASET to its path, run Cell 6.6 then Cell 7.
RESUME = False
RESUME_DATASET = "/kaggle/input/REPLACE-with-your-checkpoint-dataset"  # holds the prior runs/OUTPUT
print("config:", MODEL, "engine=", ENGINE, "base=", HF_REPO, "seq=", MAX_SEQ,
      "rank=", RANK, "targets=", TARGETS, "steps=", MAX_STEPS, "resume=", RESUME)


In [ ]:
# Cell 2 — GPU preflight (E4B needs 2×T4 for the sharded fit)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Settings → Accelerator → GPU T4 ×2"
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\n⚠️  Only 1 GPU — E4B@1664 will OOM (forward alone doesn't fit one T4).\n"
          "    Set Accelerator to GPU T4 ×2, or use the E2B notebook on a single T4.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in (f"{DATA_STEM}_train.jsonl", f"{DATA_STEM}_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 6.5 - FIT TEST (go/no-go): real Unsloth path, 3 steps, 64 rows, BOTH GPUs.
# True memory profile (the old single-GPU HF probe was heavier+misleading - ignore it).
# Survives 3 steps -> run Cell 7. OOM -> climb the Cell-1 ladder (RANK 16->8). Never cut seq.
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-seq", str(MAX_SEQ),
       "--rank", str(RANK), "--targets", TARGETS, "--grad-accum", "1",
       "--max-steps", "3", "--max-examples", "64", "--max-val", "0", "--data-stem", DATA_STEM, "--output", "fit_test"]
print(">", " ".join(cmd))
r = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
         "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
         "TORCHDYNAMO_DISABLE": "1", "UNSLOTH_COMPILE_DISABLE": "1"})
print("RC", r.returncode)
print("STDOUT tail:")
print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR tail:")
    print(r.stderr[-4000:])
    raise SystemExit("fit test FAILED - read STDERR. If CUDA OOM: RANK 16->8, re-run. "
                     "Confirm Cell 2 shows count=2 (balanced needs 2 GPUs).")
print("FITS on the 2xT4 sharded path -> run Cell 7. Note the s/step printed above.")


In [ ]:
# Cell 6.6 — RESUME prep (only when RESUME=True; skip on the first session).
# Stages the prior session's runs/OUTPUT (uploaded as a Kaggle Dataset) so Cell 7 --resume
# continues from the saved adapter + step. Works across Kaggle accounts.
import os, shutil, glob
if RESUME:
    dst = f"{WORKDIR}/runs/{OUTPUT}"
    os.makedirs(dst, exist_ok=True)
    src = RESUME_DATASET
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        print("unzip", zips[0], "->", dst); shutil.unpack_archive(zips[0], dst)
    elif os.path.isdir(src):
        for sub in ("checkpoint", "best"):
            s = os.path.join(src, sub)
            if not os.path.isdir(s):
                s = next((q for q in glob.glob(f"{src}/**/{sub}", recursive=True)), None)
            if s and os.path.isdir(s):
                shutil.copytree(s, os.path.join(dst, sub), dirs_exist_ok=True)
                print("copied", s, "->", os.path.join(dst, sub))
    assert os.path.exists(f"{dst}/checkpoint/trainer_state.pt"),         f"no checkpoint/trainer_state.pt under {dst} — check RESUME_DATASET"
    print("✅ resume checkpoint in place:", os.listdir(f"{dst}/checkpoint"))
else:
    print("RESUME=False — fresh run (first session).")


In [ ]:
# Cell 7 — train (E4B QLoRA, Unsloth, auto device_map=balanced on 2 GPUs) -> runs/<OUTPUT>
# Checkpoints adapter+optimizer every SAVE_EVERY steps; on a 12h cutoff / OOM / disconnect
# it saves before exiting so you can resume. Best (lowest val) adapter -> runs/<OUTPUT>/best.
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-steps", str(MAX_STEPS),
       "--rank", str(RANK), "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM),
       "--max-seq", str(MAX_SEQ), "--eval-every", str(EVAL_EVERY), "--max-val", str(MAX_VAL),
       "--save-every", str(SAVE_EVERY), "--data-stem", DATA_STEM, "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")
if RESUME:
    cmd.append("--resume")
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
         "TORCHDYNAMO_DISABLE": "1", "UNSLOTH_COMPILE_DISABLE": "1"})


In [ ]:
# Cell 8 — package adapter for download (run after training OR after a timeout/crash).
# runs/OUTPUT contains: best/ (lowest-val adapter -> SERVE this) + checkpoint/ (adapter +
# optimizer + step -> RESUME from this next session) + the final adapter. /kaggle/working
# persists to the notebook Output even if the 12h kernel is hard-killed, so checkpoint/
# survives a timeout. Re-upload this zip as a Kaggle Dataset to resume on any account.
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("run dir contents:", os.listdir(run_dir) if os.path.isdir(run_dir) else "MISSING")
out_zip = f"/kaggle/working/{OUTPUT}"
shutil.make_archive(out_zip, "zip", run_dir)
sz = os.path.getsize(out_zip + ".zip") / 1e6
print(f"\n✅ {out_zip}.zip ({sz:.1f} MB) — download from the Output panel.")
print("   SERVE  -> the best/ folder inside (lowest val).")
print("   RESUME -> next session: add this zip as a Dataset, set RESUME=True +")
print("             RESUME_DATASET to it, run Cells 1-6 then 6.6 then 7.")
